In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Build QA Chain
from langchain_classic.chains import RetrievalQA
from langchain_classic import hub

# Ollama LLM
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_ollama import OllamaEmbeddings

load_dotenv()

# embedding model
embedding = OllamaEmbeddings(model="nomic-embed-text")
index_name = "tax-markdown-index"

database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)

llm = ChatOllama(model="llama3:latest")

# get prompt from hub
prompt = hub.pull("rlm/rag-prompt")

# retrieve document based on query
retriever = database.as_retriever(search_kwargs={"k": 4})

# define QA Chain
qa_chain = RetrievalQA.from_chain_type(llm, retriever=retriever, chain_type_kwargs={"prompt": prompt})

# define dictionary
dictionary = ["사람을 나타내는 표현 -> 거주자"]

dictionary_prompt = ChatPromptTemplate.from_template(
    f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서, 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    변경된 질문만 출력해주세요. 다른 설명은 하지 마세요.

    사전: {dictionary}

    질문: {{question}}
"""
)

dictionary_chain = dictionary_prompt | llm | StrOutputParser()

# build tax_chain (dictionary_chain + qa_chain)
tax_chain = {"query": dictionary_chain} | qa_chain